### Create training dataset
In this notebook We are going to create training datasets and register to Hopsworks Feature Store. This training dataset will be later used to train for graph emmbedings model.
![Training Dataset](./images/create_training_dataset.png)

In [3]:
# Initialize SparkSession for local execution
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AML Training Dataset Preparation") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/02 13:47:59 WARN Utils: Your hostname, destan-do, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/02 13:47:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/02 13:48:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/02 13:48:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


In [4]:
import hashlib
import os
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import pandas as pd

# Define paths for local data
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
os.makedirs(TRAINING_DATA_PATH, exist_ok=True)

print(f"Loading feature groups from: {OUTPUT_PATH}")
print(f"Saving training datasets to: {TRAINING_DATA_PATH}")

Loading feature groups from: /home/adnoman/projects/aml_gan/AMLend2end/output
Saving training datasets to: /home/adnoman/projects/aml_gan/AMLend2end/training_data


### Instantiate a connection and get the project feature store handler 

In [5]:
# Skipping Hopsworks connection - loading from local parquet files instead
# connection = hsfs.connection()
# fs = connection.get_feature_store()

print("Loading feature groups from local parquet files (created by notebook 1)")

Loading feature groups from local parquet files (created by notebook 1)


### Get transactions feature group handle

In [6]:
# Load transactions feature group from local parquet
transactions_fg = spark.read.parquet(os.path.join(OUTPUT_PATH, "transactions_fg.parquet"))
transactions_fg.show(5)

+--------+--------+--------------+-------+-------+--------+
|  source|  target|tran_timestamp|tran_id|tx_type|base_amt|
+--------+--------+--------------+-------+-------+--------+
|3aa9646b|1e46e726|        Jan-01|    496|      4|  858.77|
|49203bc3|a74d1101|        Jan-01|   1342|      4|  386.86|
|616d4505|99af2455|        Jan-02|   1580|      4|  616.43|
|39be1ea2|e7ec7bdb|        Jan-02|   2866|      4|  146.44|
|e2e0d938|afc399a9|        Jan-03|   3997|      4|  439.09|
+--------+--------+--------------+-------+-------+--------+
only showing top 5 rows


In [7]:
transactions_fg.count()

438386

### Load alert transactions feature group  handle

In [8]:
# Load alert transactions feature group from local parquet
alert_transactions_fg = spark.read.parquet(os.path.join(OUTPUT_PATH, "alert_transactions_fg.parquet"))
alert_transactions_fg.show(5)

+--------+--------------+------+-------+
|alert_id|    alert_type|is_sar|tran_id|
+--------+--------------+------+-------+
|      47|gather_scatter|     1|  11873|
|      47|gather_scatter|     1|  11874|
|      47|gather_scatter|     1|  11875|
|      47|gather_scatter|     1|  13151|
|      47|gather_scatter|     1|  23148|
+--------+--------------+------+-------+
only showing top 5 rows


In [9]:
alert_transactions_fg.count()

915

### Load party feature group  handle

In [10]:
# Load party feature group from local parquet
party_fg = spark.read.parquet(os.path.join(OUTPUT_PATH, "party_fg.parquet"))
party_fg.show()

+--------+----+
|      id|type|
+--------+----+
|5628bd6c|   0|
|a1fcba39|   0|
|f56c9501|   1|
|9969afdd|   0|
|b356eeae|   1|
|3406706a|   0|
|26c56102|   0|
|e386ebf7|   1|
|8c094b0d|   1|
|939235aa|   1|
|de6bf2a5|   0|
|33a8ff5b|   0|
|a32807a1|   1|
|2906ef08|   0|
|c2a01b8d|   1|
|5a99160f|   1|
|8b9017b8|   0|
|fcf3bbf3|   1|
|5132aa4d|   0|
|68b90958|   1|
+--------+----+
only showing top 20 rows


## Create training datasets
To create training datasest we will use hsfs `Query` object. Training dataset's metadata, created from hsfs `Query` object, contains information such as: 
* which feature groups it was created from; 
* commit id of these feature froups;
* the order of features. 

This will give us possibility to 
* track back and see which feature were used to create this training; 
* perform time-travel and see how features looked like when this training dataset was created ;
* reconstruct feature order during model inferencing.     

### Create graph edge training datasets 

In [11]:
# Select edge features from transactions
edges = transactions_fg.select("source", "target", "tran_id", "tx_type", "base_amt")

In [12]:
edges.show()

+--------+--------+-------+-------+--------+
|  source|  target|tran_id|tx_type|base_amt|
+--------+--------+-------+-------+--------+
|3aa9646b|1e46e726|    496|      4|  858.77|
|49203bc3|a74d1101|   1342|      4|  386.86|
|616d4505|99af2455|   1580|      4|  616.43|
|39be1ea2|e7ec7bdb|   2866|      4|  146.44|
|e2e0d938|afc399a9|   3997|      4|  439.09|
|75c9a805|d7a317f6|   5518|      4|   361.0|
|c14f4989|733a496b|   7340|      4|  768.98|
|576eb672|aa49b0eb|   9376|      4|   943.4|
|847a9cf6|b070a6bb|  10362|      4|   668.3|
|12a388ff|586377aa|  10817|      4|  139.84|
|b36f9c84|1b467848|  11317|      4|  499.47|
|362e42e0|385afb8b|  11748|      4|  357.96|
|572014da|acd60eca|  13285|      4|   630.9|
|5ff2d9a7|31976e38|  14832|      4|  685.07|
|24bf603c|fcf3bbf3|  15619|      4|  964.81|
|9a118f8d|ca0967a6|  16574|      4|  919.76|
|65b8a85f|bc0de3c7|  18944|      4|  302.26|
|d7a317f6|31db9495|  19204|      4|  637.18|
|95aac0c4|3a63a8fc|  19530|      4|  630.43|
|5411dc34|

In [13]:
edges.count()

438386

In [14]:
# Save edges training dataset locally as CSV
# (replaces fs.create_training_dataset)
edges_path = os.path.join(TRAINING_DATA_PATH, "edges_td.csv")
edges.toPandas().to_csv(edges_path, index=False)
print(f"Saved edges training dataset to: {edges_path}")

Saved edges training dataset to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/edges_td.csv


### Create graph node training dataset

In [15]:
# Create nodes training dataset from party feature group
nodes = party_fg
nodes.show()

# Save nodes training dataset locally as CSV
nodes_path = os.path.join(TRAINING_DATA_PATH, "node_td.csv")
nodes.toPandas().to_csv(nodes_path, index=False)
print(f"Saved nodes training dataset to: {nodes_path}")

+--------+----+
|      id|type|
+--------+----+
|5628bd6c|   0|
|a1fcba39|   0|
|f56c9501|   1|
|9969afdd|   0|
|b356eeae|   1|
|3406706a|   0|
|26c56102|   0|
|e386ebf7|   1|
|8c094b0d|   1|
|939235aa|   1|
|de6bf2a5|   0|
|33a8ff5b|   0|
|a32807a1|   1|
|2906ef08|   0|
|c2a01b8d|   1|
|5a99160f|   1|
|8b9017b8|   0|
|fcf3bbf3|   1|
|5132aa4d|   0|
|68b90958|   1|
+--------+----+
only showing top 20 rows
Saved nodes training dataset to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/node_td.csv


## create derived feature group `alert_nodes_fg`, whether nodes were part of previously known money laundering scheme or not

In [16]:
# Create alert edges by joining transactions with alert_transactions
alert_edges = transactions_fg.select("source", "target", "tran_id", "tx_type", "base_amt") \
    .join(alert_transactions_fg.select("tran_id", "is_sar"), ["tran_id"], "left")
alert_edges = alert_edges.where(F.col("is_sar") == 1)
alert_sources = alert_edges.select("source").toDF("id")
alert_targets = alert_edges.select("target").toDF("id")
alert_nodes = alert_sources.union(alert_targets).dropDuplicates(subset=["id"])
alert_nodes = alert_nodes.withColumn("is_sar", F.lit(1))
alert_nodes.cache()
alert_nodes.show()

+--------+------+
|      id|is_sar|
+--------+------+
|33a8ff5b|     1|
|43e028ef|     1|
|fcf3bbf3|     1|
|8b9017b8|     1|
|9c187eed|     1|
|65636b63|     1|
|68c0230d|     1|
|550a25ff|     1|
|d73e5230|     1|
|c0be245b|     1|
|cdbd2ed5|     1|
|963b978f|     1|
|84563a83|     1|
|da77c74b|     1|
|840701de|     1|
|dc37f73b|     1|
|b0f4351c|     1|
|dd2ebcf1|     1|
|c29d75dc|     1|
|d7c99aa5|     1|
+--------+------+
only showing top 20 rows


In [17]:
# Duplicate cell from original - keeping for consistency
alert_sources = alert_edges.select("source").toDF("id")
alert_targets = alert_edges.select("target").toDF("id")
alert_nodes = alert_sources.union(alert_targets).dropDuplicates(subset=["id"])
alert_nodes = alert_nodes.withColumn("is_sar", F.lit(1))
alert_nodes.cache()
alert_nodes.show()

+--------+------+
|      id|is_sar|
+--------+------+
|33a8ff5b|     1|
|43e028ef|     1|
|fcf3bbf3|     1|
|8b9017b8|     1|
|9c187eed|     1|
|65636b63|     1|
|68c0230d|     1|
|550a25ff|     1|
|d73e5230|     1|
|c0be245b|     1|
|cdbd2ed5|     1|
|963b978f|     1|
|84563a83|     1|
|da77c74b|     1|
|840701de|     1|
|dc37f73b|     1|
|b0f4351c|     1|
|dd2ebcf1|     1|
|c29d75dc|     1|
|d7c99aa5|     1|
+--------+------+
only showing top 20 rows


26/02/02 13:49:00 WARN CacheManager: Asked to cache already cached data.


In [18]:
# Join nodes with alert_nodes to create alert_nodes_df
alert_nodes_df = nodes.join(alert_nodes, ["id"], "left") \
    .withColumn("is_sar", F.when(F.col("is_sar") == 1, F.col("is_sar")).otherwise(0))
alert_nodes_df.cache()
alert_nodes_df.show()

+--------+----+------+
|      id|type|is_sar|
+--------+----+------+
|5628bd6c|   0|     0|
|a1fcba39|   0|     0|
|f56c9501|   1|     0|
|9969afdd|   0|     0|
|b356eeae|   1|     0|
|3406706a|   0|     0|
|26c56102|   0|     0|
|e386ebf7|   1|     0|
|8c094b0d|   1|     0|
|939235aa|   1|     0|
|de6bf2a5|   0|     0|
|33a8ff5b|   0|     1|
|a32807a1|   1|     0|
|2906ef08|   0|     0|
|c2a01b8d|   1|     0|
|5a99160f|   1|     0|
|8b9017b8|   0|     1|
|fcf3bbf3|   1|     1|
|5132aa4d|   0|     0|
|68b90958|   1|     0|
+--------+----+------+
only showing top 20 rows


In [19]:
alert_nodes_df.where(F.col("is_sar") == 1).count()

816

In [20]:
alert_nodes_df.where(F.col("is_sar") == 0).count()

6531

In [21]:
# Save alert_nodes feature group locally
# (replaces fs.create_feature_group)
alert_nodes_path = os.path.join(OUTPUT_PATH, "alert_nodes_fg.parquet")
alert_nodes_df.toPandas().to_parquet(alert_nodes_path, index=False)
print(f"Saved alert_nodes feature group to: {alert_nodes_path}")

# Also save as CSV for training
alert_nodes_csv_path = os.path.join(TRAINING_DATA_PATH, "alert_nodes_td.csv")
alert_nodes_df.toPandas().to_csv(alert_nodes_csv_path, index=False)
print(f"Saved alert_nodes training dataset to: {alert_nodes_csv_path}")

print("\n✓ All training datasets created!")

Saved alert_nodes feature group to: /home/adnoman/projects/aml_gan/AMLend2end/output/alert_nodes_fg.parquet
Saved alert_nodes training dataset to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/alert_nodes_td.csv

✓ All training datasets created!


### Training datasets exploration from the user interface

##### Hopsworks provides user interface that enables to discover and explore avaibale Training datasets and related features. Bellow screen shot demonstrates how one can preview list of available features in `edges_td` and get basic information such us identify feature types and which one is as a label.   

![Incremental Feature Engineering](./images/td_features.png)

##### One of the important steps of training dataset exploration is discover distribution of ist features. Since we enabled statistics to be computed durring training dataset creation we can easily preview descriptive statitsics. If training dataset has splits we can also preview statitics for each split separately and campare distributions to make sure that train and test splits have similar distibutions.   


![Incremental Feature Engineering](./images/td_stats.png)

##### Hopsworks UI also give access to training dataset activity timeline metadata.  
![Incremental Feature Engineering](./images/td_activity.png)

In [ ]:
# Cleanup: Stop Spark session before closing notebook
# Run this cell when done to free resources for other notebooks
spark.stop()
print("Spark session stopped - resources freed")